In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Global Master Search:      23504
   Global Master Search Data: 1400
   Search Artists:            22104
   Errors:                    85
   Known Summary IDs:         0


# Search For New Artists

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [33]:
from musicdb import PanDBIO
from gate import IOStore

pdbio = PanDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

ios     = IOStore()
mdbio   = ios.get(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

del pdbio
del mmeDF
del refData
del mbIDData

#   Artist Names To Get:          793373

SetListFM Search Results
   Known Master Artist Names:    814494
   Known Spotify Artist Names:   30071
   Artist Names To Get:          784337


## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

## Run @ 3-4 PM every day

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(20)
    n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM ArtistIDs] Start    ==> Time Is 2022-04-25 15:05:35
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-04-26 11:00:00 <====
   ====> Will Terminate Process 19 Hours and 54 Minutes From Now
Searching For Lionel Loueke (7aa056f7-9a57-4cdc-b856-6aa12c98624f)                            True
Searching For Heather B (4af98b74-33f8-4010-b58e-20c6f7cf1bbb)                                True
Searching For Sinan Özen (15fa280f-5033-4eaa-826c-97903af06fdd)                              True
Searching For Bernice (8ff18a59-05df-4214-a9de-08deda8ec3df)                                  True
Searching For B.R.O. (038ab402-f51c-40cf-b6b6-9e08a1c1554a)                                   True
5/?        : Process [Getting SetListFM ArtistIDs] Has Run For 1 Minute and 44 Seconds.
Saving 30076 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Benny Scott (a1cd1bac-4f71-4a22-b789-fe4038d89269)                              True
Searching For Sir Chloe (3c3a1ea4-922a-4a90-b1aa-b7763d6dbf20)                                True
Searching For Zero T (2930b614-d54b-4d29-87fe-83ac9c9fdbda)                                   True
Searching For La Arrolladora Banda el Limón de René Camacho (91398a44-7656-4e66-9ce7-caf3ca4318d8)True
Searching For Intense (9e7ec35b-b0cb-406e-9ec4-7b6c62c57197)                                  True
10/?       : Process [Getting SetListFM ArtistIDs] Has Run For 3 Minutes and 38 Seconds.
Saving 30081 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Jenny Morris (d5f326c9-6e03-43d0-9397-7e7e9a7b7b93)                             True
Searching For Mad Dog Loose (89a720b2-2e4f-471d-bf89-18a1534023ea)                            True
Searching For A Broken Consort (71013a6c-f9b1-41ac-b68b-a1549c846795)                         True
Searching For Dub Syndicate (302ae60e-4c6f-444b-9a1f-8aa5f3aed57e)                            True
Searching For O-Town (5b5b5410-29d7-478a-94ca-bffa06c73579)                                   True
15/?       : Process [Getting SetListFM ArtistIDs] Has Run For 5 Minutes and 36 Seconds.
Saving 30086 SetListFM Searched For Artist IDs


...

Searching For Nico Morano (7888bad2-7fa2-45e4-b99d-a424db188581)                              True
Searching For Greg Howe (63d55bc3-62d6-4005-b9d5-c06f6e84451a)                                True
Searching For Grupo Montéz de Durango (337609a2-6b4b-45ee-822c-62b7f727bca8)                 True
Searching For MEG (b3b665a4-5a57-40f3-9df3-cdbb4e9ade59)                                      True
Searching For Broery Marantika (162a9905-875f-4c9c-81a8-fb5a5a7740d5)                         True
825/?      : Process [Getting SetListFM ArtistIDs] Has Run For 5 Hours and 17 Minutes.
Saving 30896 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For National Orchestra of Belgium (dad46109-6ad6-4c61-a493-580049978b41)            True
Searching For Seisei (25c72ce8-dd13-4892-a27a-afe198a59faa)                                   True
Searching For P.E.A.C.E. (87bb0540-12e8-4318-a622-b30fb78baf50)                               True
Searching For Otis Williams And The Charms (30e81941-562a-4342-9d62-246cc9cf29bd)             True
Searching For Daniela Araújo (418816a9-82da-4c5a-acb0-0e2768761b34)                          True
830/?      : Process [Getting SetListFM ArtistIDs] Has Run For 5 Hours and 19 Minutes.
Saving 30901 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For arbour (af0f34ae-01f3-4c65-b9b8-934964312c96)                                   True
Searching For Daniel Bovie (cde8fe78-e697-45ed-8038-3181433da78d)                             

In [24]:

del searchedForErrors['260523060702005307194169312891497960966']
del searchedForErrors['204688363120401809827902113017497819567']
del searchedForErrors['52601397499669132316615570594880471136']
del searchedForErrors['100289869541857586668960815138953544847']
del searchedForErrors['155129401683092282537119538701124448454']
del searchedForErrors['30043587855655597963313294794338118145']
errors.save(data=searchedForErrors)

## Save Results

In [30]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [31]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 1400 New Artists
Found 28671 Previous Artists
Found 30071 Total Results
Found 30071 Unique Results
Saving Data


In [32]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
mio.data.getRawArtistInfoData(mio.mv.get(3540473060), 3540473060)

In [ ]:
localArtistsDict

In [ ]:
localArtists.save(data=localArtistsDict)


In [ ]:
tracks
disc_count